In [1]:
import kagglehub
import pandas as pd
import os

# Download latest version to ensure 'path' is defined
path = kagglehub.dataset_download("yasiralifarrukh/unsw-and-cicids2017-labelled-pcap-data")

print("Path to dataset files:", path)

# List files in the downloaded directory to check available filenames
print(f"\nAvailable files in {path}: {os.listdir(path)}")

try:
    # Correcting the CSV filenames based on available files
    # Load only a sample of rows to check for memory issues
    df_unsw = pd.read_csv(os.path.join(path, 'Payload_data_UNSW.csv'), nrows=10000)
    df_cicids = pd.read_csv(os.path.join(path, 'Payload_data_CICIDS2017.csv'), nrows=10000)

    print("\n--- Top 10 rows of Payload_data_UNSW.csv (sample) ---")
    display(df_unsw.head(10))

    print("\n--- Top 10 rows of Payload_data_CICIDS2017.csv (sample) ---")
    display(df_cicids.head(10))

except FileNotFoundError:
    print(f"Could not find one or both CSV files in the path: {path}")
    print("Please check the exact filenames and try again based on the listed files above.")
except Exception as e:
    print(f"An error occurred: {e}")

100%|██████████| 330M/330M [00:15<00:00, 21.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/yasiralifarrukh/unsw-and-cicids2017-labelled-pcap-data/versions/1

Available files in /root/.cache/kagglehub/datasets/yasiralifarrukh/unsw-and-cicids2017-labelled-pcap-data/versions/1: ['Payload_data_CICIDS2017.csv', 'Payload_data_UNSW.csv']

--- Top 10 rows of Payload_data_UNSW.csv (sample) ---


,payload_byte_1,payload_byte_2,payload_byte_3,payload_byte_4,payload_byte_5,payload_byte_6,payload_byte_7,payload_byte_8,payload_byte_9,payload_byte_10,...,payload_byte_1496,payload_byte_1497,payload_byte_1498,payload_byte_1499,payload_byte_1500,ttl,total_len,protocol,t_delta,label
0,66,80,83,33,32,55,98,32,66,80,...,0,0,0,0,0,254,100,others,0.000010,analysis
1,66,80,83,33,32,53,52,32,66,80,...,0,0,0,0,0,254,100,others,0.000005,analysis
2,66,80,83,33,32,57,32,66,80,83,...,0,0,0,0,0,254,90,others,0.000008,analysis
3,66,80,83,33,32,56,51,32,66,80,...,0,0,0,0,0,254,100,pipe,0.000004,analysis
4,66,80,83,33,32,57,32,66,80,83,...,0,0,0,0,0,254,90,others,0.000003,analysis
5,66,80,83,33,32,57,32,66,80,83,...,0,0,0,0,0,254,90,others,0.000006,analysis
6,66,80,83,33,32,53,101,32,66,80,...,0,0,0,0,0,254,100,ipip,0.000009,analysis
7,66,80,83,33,32,52,49,32,66,80,...,0,0,0,0,0,254,100,others,0.000005,analysis
8,66,80,83,33,32,53,55,32,66,80,...,0,0,0,0,0,254,100,others,0.000005,analysis
9,66,80,83,33,32,51,102,32,66,80,...,0,0,0,0,0,254,100,others,0.000009,analysis



--- Top 10 rows of Payload_data_CICIDS2017.csv (sample) ---


,payload_byte_1,payload_byte_2,payload_byte_3,payload_byte_4,payload_byte_5,payload_byte_6,payload_byte_7,payload_byte_8,payload_byte_9,payload_byte_10,...,payload_byte_1496,payload_byte_1497,payload_byte_1498,payload_byte_1499,payload_byte_1500,ttl,total_len,protocol,t_delta,label
0,43,27,1,16,0,1,0,0,0,0,...,0,0,0,0,0,64,78,udp,0.085168,BENIGN
1,23,40,1,0,0,1,0,0,0,0,...,0,0,0,0,0,128,77,udp,0.000052,BENIGN
2,73,58,95,1,125,198,144,14,65,220,...,0,0,0,0,0,54,1210,tcp,0.000217,BENIGN
3,244,132,1,0,0,1,0,0,0,0,...,0,0,0,0,0,128,63,udp,0.000003,BENIGN
4,124,208,100,115,11,136,116,143,223,234,...,0,0,0,0,0,83,1450,tcp,0.000203,BENIGN
5,254,218,1,0,0,1,0,0,0,0,...,0,0,0,0,0,128,85,udp,0.000175,BENIGN
6,170,226,248,161,225,220,6,163,216,229,...,0,0,0,0,0,57,1500,tcp,0.000187,BENIGN
7,213,34,1,0,0,1,0,0,0,0,...,0,0,0,0,0,128,81,udp,0.000001,BENIGN
8,227,2,6,234,0,0,17,89,0,0,...,0,0,0,0,0,64,76,udp,0.008723,BENIGN
9,109,226,227,240,141,4,82,196,121,241,...,0,0,0,0,0,57,1500,tcp,0.000167,BENIGN


In [2]:
import numpy as np
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import pickle
from google.colab import files
import pandas as pd # Ensure pandas is imported for DataFrame operations
import os # Ensure os is imported for path operations
import kagglehub # Ensure kagglehub is imported for dataset download
import gc # Import garbage collector

### 1. Load and balance the datasets

First, we'll reload the datasets, loading up to 80,000 rows from each. Then, we'll clean the labels to categorize them as 'Normal' (0) or 'Attack' (1) and sample them to create a balanced training set.

In [3]:
# Reload data with up to 80,000 rows (or fewer if the file is smaller) to ensure enough samples for balancing
# The 'path' variable is already defined from the previous execution of cell a89738b4
# path = kagglehub.dataset_download("yasiralifarrukh/unsw-and-cicids2017-labelled-pcap-data") # No need to re-download
df_unsw = pd.read_csv(os.path.join(path, 'Payload_data_UNSW.csv'), nrows=80000)
df_cicids = pd.read_csv(os.path.join(path, 'Payload_data_CICIDS2017.csv'), nrows=80000)

# Free up memory after loading raw data if it's no longer needed in its original form
gc.collect()

# Function to clean labels
def clean_labels(df):
    # Convert all labels to lowercase for consistent processing
    df['label'] = df['label'].astype(str).str.lower()
    # Map 'normal' or 'benign' to 0, and all other labels to 1
    df['attack_label'] = df['label'].apply(lambda x: 0 if x in ['normal', 'benign'] else 1)
    return df

df_unsw = clean_labels(df_unsw)
df_cicids = clean_labels(df_cicids)

print("UNSW Dataset Label Distribution after cleaning:")
print(df_unsw['attack_label'].value_counts())
print("\nCICIDS Dataset Label Distribution after cleaning:")
print(df_cicids['attack_label'].value_counts())

UNSW Dataset Label Distribution after cleaning:
attack_label
1    58881
0    21000
Name: count, dtype: int64

CICIDS Dataset Label Distribution after cleaning:
attack_label
0    80000
Name: count, dtype: int64


In [4]:
# Sample equal portions for a balanced training set
def balance_dataset(df, normal_samples=40000, attack_samples=40000):
    df_normal = df[df['attack_label'] == 0]
    df_attack = df[df['attack_label'] == 1]

    # Sample with replacement if there aren't enough samples
    normal_sampled = df_normal.sample(min(normal_samples, len(df_normal)), random_state=42, replace=True)
    attack_sampled = df_attack.sample(min(attack_samples, len(df_attack)), random_state=42, replace=True)

    return pd.concat([normal_sampled, attack_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)

df_unsw_balanced = balance_dataset(df_unsw, normal_samples=0, attack_samples=40000) # UNSW primarily has attacks
df_cicids_balanced = balance_dataset(df_cicids, normal_samples=40000, attack_samples=0) # CICIDS primarily has normal

# Combine the balanced datasets to create an 80,000 record training set
df_combined = pd.concat([df_unsw_balanced, df_cicids_balanced]).sample(frac=1, random_state=42).reset_index(drop=True)

# Clear original dataframes from memory
del df_unsw, df_cicids, df_unsw_balanced, df_cicids_balanced
gc.collect()

print("\nCombined Dataset Label Distribution (total 80,000 records):")
print(df_combined['attack_label'].value_counts())
display(df_combined.head())


Combined Dataset Label Distribution (total 80,000 records):
attack_label
0    40000
1    40000
Name: count, dtype: int64


,payload_byte_1,payload_byte_2,payload_byte_3,payload_byte_4,payload_byte_5,payload_byte_6,payload_byte_7,payload_byte_8,payload_byte_9,payload_byte_10,...,payload_byte_1497,payload_byte_1498,payload_byte_1499,payload_byte_1500,ttl,total_len,protocol,t_delta,label,attack_label
0,167,214,1,0,0,1,0,0,0,0,...,0,0,0,0,128,77,udp,0.000004,benign,0
1,23,3,3,0,33,0,0,0,0,0,...,0,0,0,0,128,78,tcp,0.000130,benign,0
2,0,3,244,128,0,6,0,0,192,168,...,0,0,0,0,64,90,udp,0.000007,benign,0
3,206,230,1,0,0,1,0,0,0,0,...,0,0,0,0,128,68,udp,0.000961,benign,0
4,211,251,127,206,136,195,255,193,19,63,...,47,190,77,112,51,1794,tcp,0.000103,benign,0


### 2. Clean and preprocess features

Next, we'll drop metadata columns, map protocol values, and scale the features using `StandardScaler`.

### 2. Clean and preprocess features (Updated for 80,000 records)

Next, we'll drop metadata columns, map protocol values, and scale the features using `StandardScaler`. This step now processes the full 80,000 record dataset.

In [5]:
metadata_columns_to_drop = [
    'id', 'srcip', 'dstip', 'sport', 'dsport', 'stime', 'ltime',
    'Stime', 'Ltime', 'src_ip', 'dst_ip', 't_delta'
]

# Filter to include only columns that exist in df_combined
actual_columns_to_drop = [col for col in metadata_columns_to_drop if col in df_combined.columns]

df_processed = df_combined.drop(columns=actual_columns_to_drop, errors='ignore')

# Handle 'protocol' column: map values and convert to numeric
# First, fill any potential NaN values to prevent errors during string operations
df_processed['protocol'] = df_processed['protocol'].fillna('unknown').astype(str).str.lower()

protocol_mapping = {
    'tcp': 6.0, '6': 6.0,
    'udp': 17.0, '17': 17.0,
    'icmp': 1.0, '1': 1.0
}

def map_protocol_to_numeric(protocol_str):
    if protocol_str in protocol_mapping:
        return protocol_mapping[protocol_str]
    try:
        # Try converting directly to float if it's a numeric string not in map
        return float(protocol_str)
    except ValueError:
        # Assign a consistent numeric value for other string protocols not in mapping.
        # This ensures all protocol values are numeric for scaling.
        return 99.0

# Apply mapping to the protocol column
df_processed['protocol'] = df_processed['protocol'].apply(map_protocol_to_numeric)


# Separate features (X) and labels (y)
X = df_processed.drop(columns=['label', 'attack_label'])
y = df_processed['attack_label']

# Store feature column names for later use
feature_columns = X.columns.tolist()

# Scale the features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Shape of features after preprocessing: {X_scaled.shape}")
print(f"Number of features: {len(feature_columns)}")
print("First 5 rows of scaled features (sample):")
print(X_scaled[:5])

Shape of features after preprocessing: (80000, 1503)
Number of features: 1503
First 5 rows of scaled features (sample):
[[ 1.07399036  1.75656384 -0.7302174  ... -0.473423   -0.52148541
  -0.0314382 ]
 [-0.89842643 -1.01992409 -0.70043163 ... -0.473423   -0.52032358
  -0.51430923]
 [-1.21346523 -1.01992409  2.88875402 ... -1.23796633 -0.50638168
  -0.0314382 ]
 [ 1.60818658  1.96710321 -0.7302174  ... -0.473423   -0.53194183
  -0.0314382 ]
 [ 1.67667327  2.24343613  1.1462863  ... -1.3932642   1.47336838
  -0.51430923]]


In [6]:
# Re-executing preprocessing to ensure X_scaled and y reflect the 80,000-record df_combined

# Define columns to drop (metadata/IP columns) based on user's request.
# Only include those actually present in the dataframe to avoid errors.
metadata_columns_to_drop = [
    'id', 'srcip', 'dstip', 'sport', 'dsport', 'stime', 'ltime',
    'Stime', 'Ltime', 'src_ip', 'dst_ip', 't_delta'
]

# Filter to include only columns that exist in df_combined
actual_columns_to_drop = [col for col in metadata_columns_to_drop if col in df_combined.columns]

df_processed = df_combined.drop(columns=actual_columns_to_drop, errors='ignore')

# Handle 'protocol' column: map values and convert to numeric
# First, fill any potential NaN values to prevent errors during string operations
df_processed['protocol'] = df_processed['protocol'].fillna('unknown').astype(str).str.lower()

protocol_mapping = {
    'tcp': 6.0, '6': 6.0,
    'udp': 17.0, '17': 17.0,
    'icmp': 1.0, '1': 1.0
}

def map_protocol_to_numeric(protocol_str):
    if protocol_str in protocol_mapping:
        return protocol_mapping[protocol_str]
    try:
        # Try converting directly to float if it's a numeric string not in map
        return float(protocol_str)
    except ValueError:
        # Assign a consistent numeric value for other string protocols not in mapping.
        # This ensures all protocol values are numeric for scaling.
        return 99.0

# Apply mapping to the protocol column
df_processed['protocol'] = df_processed['protocol'].apply(map_protocol_to_numeric)


# Separate features (X) and labels (y)
X = df_processed.drop(columns=['label', 'attack_label'])
y = df_processed['attack_label']

# Store feature column names for later use
feature_columns = X.columns.tolist()

# Scale the features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Shape of features after preprocessing: {X_scaled.shape}")
print(f"Number of features: {len(feature_columns)}")
print("First 5 rows of scaled features (sample):")
print(X_scaled[:5])

Shape of features after preprocessing: (80000, 1503)
Number of features: 1503
First 5 rows of scaled features (sample):
[[ 1.07399036  1.75656384 -0.7302174  ... -0.473423   -0.52148541
  -0.0314382 ]
 [-0.89842643 -1.01992409 -0.70043163 ... -0.473423   -0.52032358
  -0.51430923]
 [-1.21346523 -1.01992409  2.88875402 ... -1.23796633 -0.50638168
  -0.0314382 ]
 [ 1.60818658  1.96710321 -0.7302174  ... -0.473423   -0.53194183
  -0.0314382 ]
 [ 1.67667327  2.24343613  1.1462863  ... -1.3932642   1.47336838
  -0.51430923]]


In [7]:
# Re-executing Isolation Forest training with updated hyperparameters on the full dataset
# (Cell 214d30db)
# Train Isolation Forest (Anomaly Detection)
print("\nTraining Isolation Forest...")
# Update contamination based on the balanced dataset (40k normal, 40k attack => 0.5 anomaly ratio)
isolation_forest = IsolationForest(contamination=0.5, random_state=42, n_jobs=-1) # n_jobs=-1 for parallel processing
isolation_forest.fit(X_scaled)
print("Isolation Forest training complete.")

# Isolation Forest predictions (returns -1 for anomalies, 1 for inliers)
# Convert to 1 for anomaly, 0 for normal for consistency with attack_label
isolation_forest_predictions = np.array([1 if x == -1 else 0 for x in isolation_forest.predict(X_scaled)])

print("\nIsolation Forest Anomaly Detection Report:")
print(classification_report(y, isolation_forest_predictions))
print(f"Accuracy: {accuracy_score(y, isolation_forest_predictions):.4f}")


Training Isolation Forest...
Isolation Forest training complete.

Isolation Forest Anomaly Detection Report:
              precision    recall  f1-score   support

           0       0.44      0.44      0.44     40000
           1       0.44      0.44      0.44     40000

    accuracy                           0.44     80000
   macro avg       0.44      0.44      0.44     80000
weighted avg       0.44      0.44      0.44     80000

Accuracy: 0.4440


In [8]:
# Re-executing Random Forest training with updated hyperparameters on the full dataset
# (Cell 2c7de6fd)
# Train Random Forest Classifier (Attack Classifier)
print("\nTraining Random Forest Classifier (initial for feature importance)...")
# Update hyperparameters for better accuracy and for feature importance calculation
random_forest = RandomForestClassifier(n_estimators=150, max_depth=15, min_samples_split=2, random_state=42, n_jobs=-1) # n_jobs=-1 for parallel processing
random_forest.fit(X_scaled, y)
print("Random Forest Classifier training complete.")

# Random Forest predictions
random_forest_predictions = random_forest.predict(X_scaled)

print("\nRandom Forest Classifier Report (initial):")
print(classification_report(y, random_forest_predictions))
print(f"Accuracy: {accuracy_score(y, random_forest_predictions):.4f}")


Training Random Forest Classifier (initial for feature importance)...
Random Forest Classifier training complete.

Random Forest Classifier Report (initial):
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     40000
           1       0.99      1.00      0.99     40000

    accuracy                           0.99     80000
   macro avg       0.99      0.99      0.99     80000
weighted avg       0.99      0.99      0.99     80000

Accuracy: 0.9936


In [9]:
# Re-executing feature importance and selection on the full dataset
# (Cell ef1d733f)
# Get feature importances from the trained Random Forest model
importances = random_forest.feature_importances_
feature_names = X.columns

# Create a Series for easier sorting and filtering
feature_importances = pd.Series(importances, index=feature_names)

# Sort features by importance in descending order
sorted_features = feature_importances.sort_values(ascending=False)

print("Top 20 Most Important Features:")
print(sorted_features.head(20))

# Filter out features with 0 importance
active_features = sorted_features[sorted_features > 0].index.tolist()

print(f"\nNumber of active features (importance > 0): {len(active_features)}")
print("Active features will be used for retraining models.")

# Filter X to include only active features
X_selected = X[active_features]

# Re-scale features using a new scaler for only the active features
scaler_selected = StandardScaler()
X_selected_scaled = scaler_selected.fit_transform(X_selected)

print(f"Shape of selected features after scaling: {X_selected_scaled.shape}")

# Store selected feature column names for later use
selected_feature_columns = active_features

# Clear original large X_scaled to free memory
del X_scaled
gc.collect()

Top 20 Most Important Features:
ttl                0.107685
payload_byte_21    0.032193
payload_byte_28    0.028757
payload_byte_25    0.023394
payload_byte_17    0.023046
payload_byte_6     0.021516
payload_byte_20    0.019236
payload_byte_26    0.019014
total_len          0.018089
payload_byte_22    0.017570
payload_byte_3     0.017085
payload_byte_23    0.016694
payload_byte_32    0.016449
payload_byte_29    0.016389
payload_byte_30    0.016271
payload_byte_33    0.015966
payload_byte_7     0.015578
payload_byte_9     0.013083
payload_byte_14    0.013020
payload_byte_27    0.012731
dtype: float64

Number of active features (importance > 0): 1503
Active features will be used for retraining models.
Shape of selected features after scaling: (80000, 1503)


72

In [10]:
# Re-executing refined Isolation Forest training with selected features on the full dataset
# (Cell f8338129)
# Retrain Isolation Forest with selected features
print("\nRetraining Isolation Forest with selected features...")
isolation_forest_refined = IsolationForest(contamination=0.5, random_state=42, n_jobs=-1)
isolation_forest_refined.fit(X_selected_scaled)
print("Refined Isolation Forest training complete.")

isolation_forest_predictions_refined = np.array([1 if x == -1 else 0 for x in isolation_forest_refined.predict(X_selected_scaled)])

print("\nRefined Isolation Forest Anomaly Detection Report:")
print(classification_report(y, isolation_forest_predictions_refined))
print(f"Accuracy: {accuracy_score(y, isolation_forest_predictions_refined):.4f}")


Retraining Isolation Forest with selected features...
Refined Isolation Forest training complete.

Refined Isolation Forest Anomaly Detection Report:
              precision    recall  f1-score   support

           0       0.47      0.47      0.47     40000
           1       0.47      0.47      0.47     40000

    accuracy                           0.47     80000
   macro avg       0.47      0.47      0.47     80000
weighted avg       0.47      0.47      0.47     80000

Accuracy: 0.4706


In [11]:
# Re-executing refined Random Forest training with selected features on the full dataset
# (Cell 0fdaa707)
# Retrain Random Forest Classifier with selected features
print("\nRetraining Random Forest Classifier with selected features...")
random_forest_refined = RandomForestClassifier(n_estimators=150, max_depth=15, min_samples_split=2, random_state=42, n_jobs=-1)
random_forest_refined.fit(X_selected_scaled, y)
print("Refined Random Forest Classifier training complete.")

random_forest_predictions_refined = random_forest_refined.predict(X_selected_scaled)

print("\nRefined Random Forest Classifier Report:")
print(classification_report(y, random_forest_predictions_refined))
print(f"Accuracy: {accuracy_score(y, random_forest_predictions_refined):.4f}")


Retraining Random Forest Classifier with selected features...
Refined Random Forest Classifier training complete.

Refined Random Forest Classifier Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     40000
           1       0.99      1.00      0.99     40000

    accuracy                           0.99     80000
   macro avg       0.99      0.99      0.99     80000
weighted avg       0.99      0.99      0.99     80000

Accuracy: 0.9931


In [12]:
# Re-executing preprocessing to ensure X_scaled and y reflect the 80,000-record df_combined

# Define columns to drop (metadata/IP columns) based on user's request.
# Only include those actually present in the dataframe to avoid errors.
metadata_columns_to_drop = [
    'id', 'srcip', 'dstip', 'sport', 'dsport', 'stime', 'ltime',
    'Stime', 'Ltime', 'src_ip', 'dst_ip', 't_delta'
]

# Filter to include only columns that exist in df_combined
actual_columns_to_drop = [col for col in metadata_columns_to_drop if col in df_combined.columns]

df_processed = df_combined.drop(columns=actual_columns_to_drop, errors='ignore')

# Handle 'protocol' column: map values and convert to numeric
# First, fill any potential NaN values to prevent errors during string operations
df_processed['protocol'] = df_processed['protocol'].fillna('unknown').astype(str).str.lower()

protocol_mapping = {
    'tcp': 6.0, '6': 6.0,
    'udp': 17.0, '17': 17.0,
    'icmp': 1.0, '1': 1.0
}

def map_protocol_to_numeric(protocol_str):
    if protocol_str in protocol_mapping:
        return protocol_mapping[protocol_str]
    try:
        # Try converting directly to float if it's a numeric string not in map
        return float(protocol_str)
    except ValueError:
        # Assign a consistent numeric value for other string protocols not in mapping.
        # This ensures all protocol values are numeric for scaling.
        return 99.0

# Apply mapping to the protocol column
df_processed['protocol'] = df_processed['protocol'].apply(map_protocol_to_numeric)


# Separate features (X) and labels (y)
X = df_processed.drop(columns=['label', 'attack_label'])
y = df_processed['attack_label']

# Store feature column names for later use
feature_columns = X.columns.tolist()

# Scale the features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Shape of features after preprocessing: {X_scaled.shape}")
print(f"Number of features: {len(feature_columns)}")
print("First 5 rows of scaled features (sample):")
print(X_scaled[:5])

Shape of features after preprocessing: (80000, 1503)
Number of features: 1503
First 5 rows of scaled features (sample):
[[ 1.07399036  1.75656384 -0.7302174  ... -0.473423   -0.52148541
  -0.0314382 ]
 [-0.89842643 -1.01992409 -0.70043163 ... -0.473423   -0.52032358
  -0.51430923]
 [-1.21346523 -1.01992409  2.88875402 ... -1.23796633 -0.50638168
  -0.0314382 ]
 [ 1.60818658  1.96710321 -0.7302174  ... -0.473423   -0.53194183
  -0.0314382 ]
 [ 1.67667327  2.24343613  1.1462863  ... -1.3932642   1.47336838
  -0.51430923]]


In [13]:
# Re-executing Isolation Forest training with updated hyperparameters on the full dataset
# (Cell 214d30db)
# Train Isolation Forest (Anomaly Detection)
print("\nTraining Isolation Forest...")
# Update contamination based on the balanced dataset (40k normal, 40k attack => 0.5 anomaly ratio)
isolation_forest = IsolationForest(contamination=0.5, random_state=42, n_jobs=-1) # n_jobs=-1 for parallel processing
isolation_forest.fit(X_scaled)
print("Isolation Forest training complete.")

# Isolation Forest predictions (returns -1 for anomalies, 1 for inliers)
# Convert to 1 for anomaly, 0 for normal for consistency with attack_label
isolation_forest_predictions = np.array([1 if x == -1 else 0 for x in isolation_forest.predict(X_scaled)])

print("\nIsolation Forest Anomaly Detection Report:")
print(classification_report(y, isolation_forest_predictions))
print(f"Accuracy: {accuracy_score(y, isolation_forest_predictions):.4f}")


Training Isolation Forest...
Isolation Forest training complete.

Isolation Forest Anomaly Detection Report:
              precision    recall  f1-score   support

           0       0.44      0.44      0.44     40000
           1       0.44      0.44      0.44     40000

    accuracy                           0.44     80000
   macro avg       0.44      0.44      0.44     80000
weighted avg       0.44      0.44      0.44     80000

Accuracy: 0.4440


In [14]:
# Re-executing Random Forest training with updated hyperparameters on the full dataset
# (Cell 2c7de6fd)
# Train Random Forest Classifier (Attack Classifier)
print("\nTraining Random Forest Classifier (initial for feature importance)...")
# Update hyperparameters for better accuracy and for feature importance calculation
random_forest = RandomForestClassifier(n_estimators=150, max_depth=15, min_samples_split=2, random_state=42, n_jobs=-1) # n_jobs=-1 for parallel processing
random_forest.fit(X_scaled, y)
print("Random Forest Classifier training complete.")

# Random Forest predictions
random_forest_predictions = random_forest.predict(X_scaled)

print("\nRandom Forest Classifier Report (initial):")
print(classification_report(y, random_forest_predictions))
print(f"Accuracy: {accuracy_score(y, random_forest_predictions):.4f}")


Training Random Forest Classifier (initial for feature importance)...
Random Forest Classifier training complete.

Random Forest Classifier Report (initial):
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     40000
           1       0.99      1.00      0.99     40000

    accuracy                           0.99     80000
   macro avg       0.99      0.99      0.99     80000
weighted avg       0.99      0.99      0.99     80000

Accuracy: 0.9936


In [15]:
# Re-executing feature importance and selection on the full dataset
# (Cell ef1d733f)
# Get feature importances from the trained Random Forest model
importances = random_forest.feature_importances_
feature_names = X.columns

# Create a Series for easier sorting and filtering
feature_importances = pd.Series(importances, index=feature_names)

# Sort features by importance in descending order
sorted_features = feature_importances.sort_values(ascending=False)

print("Top 20 Most Important Features:")
print(sorted_features.head(20))

# Filter out features with 0 importance
active_features = sorted_features[sorted_features > 0].index.tolist()

print(f"\nNumber of active features (importance > 0): {len(active_features)}")
print("Active features will be used for retraining models.")

# Filter X to include only active features
X_selected = X[active_features]

# Re-scale features using a new scaler for only the active features
scaler_selected = StandardScaler()
X_selected_scaled = scaler_selected.fit_transform(X_selected)

print(f"Shape of selected features after scaling: {X_selected_scaled.shape}")

# Store selected feature column names for later use
selected_feature_columns = active_features

# Clear original large X_scaled to free memory
del X_scaled
gc.collect()

Top 20 Most Important Features:
ttl                0.107685
payload_byte_21    0.032193
payload_byte_28    0.028757
payload_byte_25    0.023394
payload_byte_17    0.023046
payload_byte_6     0.021516
payload_byte_20    0.019236
payload_byte_26    0.019014
total_len          0.018089
payload_byte_22    0.017570
payload_byte_3     0.017085
payload_byte_23    0.016694
payload_byte_32    0.016449
payload_byte_29    0.016389
payload_byte_30    0.016271
payload_byte_33    0.015966
payload_byte_7     0.015578
payload_byte_9     0.013083
payload_byte_14    0.013020
payload_byte_27    0.012731
dtype: float64

Number of active features (importance > 0): 1503
Active features will be used for retraining models.
Shape of selected features after scaling: (80000, 1503)


120

In [16]:
# Re-executing refined Isolation Forest training with selected features on the full dataset
# (Cell f8338129)
# Retrain Isolation Forest with selected features
print("\nRetraining Isolation Forest with selected features...")
isolation_forest_refined = IsolationForest(contamination=0.5, random_state=42, n_jobs=-1)
isolation_forest_refined.fit(X_selected_scaled)
print("Refined Isolation Forest training complete.")

isolation_forest_predictions_refined = np.array([1 if x == -1 else 0 for x in isolation_forest_refined.predict(X_selected_scaled)])

print("\nRefined Isolation Forest Anomaly Detection Report:")
print(classification_report(y, isolation_forest_predictions_refined))
print(f"Accuracy: {accuracy_score(y, isolation_forest_predictions_refined):.4f}")


Retraining Isolation Forest with selected features...
Refined Isolation Forest training complete.

Refined Isolation Forest Anomaly Detection Report:
              precision    recall  f1-score   support

           0       0.47      0.47      0.47     40000
           1       0.47      0.47      0.47     40000

    accuracy                           0.47     80000
   macro avg       0.47      0.47      0.47     80000
weighted avg       0.47      0.47      0.47     80000

Accuracy: 0.4706


In [17]:
# Re-executing refined Random Forest training with selected features on the full dataset
# (Cell 0fdaa707)
# Retrain Random Forest Classifier with selected features
print("\nRetraining Random Forest Classifier with selected features...")
random_forest_refined = RandomForestClassifier(n_estimators=150, max_depth=15, min_samples_split=2, random_state=42, n_jobs=-1)
random_forest_refined.fit(X_selected_scaled, y)
print("Refined Random Forest Classifier training complete.")

random_forest_predictions_refined = random_forest_refined.predict(X_selected_scaled)

print("\nRefined Random Forest Classifier Report:")
print(classification_report(y, random_forest_predictions_refined))
print(f"Accuracy: {accuracy_score(y, random_forest_predictions_refined):.4f}")


Retraining Random Forest Classifier with selected features...
Refined Random Forest Classifier training complete.

Refined Random Forest Classifier Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     40000
           1       0.99      1.00      0.99     40000

    accuracy                           0.99     80000
   macro avg       0.99      0.99      0.99     80000
weighted avg       0.99      0.99      0.99     80000

Accuracy: 0.9931


In [18]:
# Re-executing preprocessing to ensure X_scaled and y reflect the 80,000-record df_combined
# This cell's content is as previously defined in 'modify_cells' of turn_2, but it needs re-execution.

# Define columns to drop (metadata/IP columns) based on user's request.
# Only include those actually present in the dataframe to avoid errors.
metadata_columns_to_drop = [
    'id', 'srcip', 'dstip', 'sport', 'dsport', 'stime', 'ltime',
    'Stime', 'Ltime', 'src_ip', 'dst_ip', 't_delta'
]

# Filter to include only columns that exist in df_combined
actual_columns_to_drop = [col for col in metadata_columns_to_drop if col in df_combined.columns]

df_processed = df_combined.drop(columns=actual_columns_to_drop, errors='ignore')

# Handle 'protocol' column: map values and convert to numeric
# First, fill any potential NaN values to prevent errors during string operations
df_processed['protocol'] = df_processed['protocol'].fillna('unknown').astype(str).str.lower()

protocol_mapping = {
    'tcp': 6.0, '6': 6.0,
    'udp': 17.0, '17': 17.0,
    'icmp': 1.0, '1': 1.0
}

def map_protocol_to_numeric(protocol_str):
    if protocol_str in protocol_mapping:
        return protocol_mapping[protocol_str]
    try:
        # Try converting directly to float if it's a numeric string not in map
        return float(protocol_str)
    except ValueError:
        # Assign a consistent numeric value for other string protocols not in mapping.
        # This ensures all protocol values are numeric for scaling.
        return 99.0

# Apply mapping to the protocol column
df_processed['protocol'] = df_processed['protocol'].apply(map_protocol_to_numeric)


# Separate features (X) and labels (y)
X = df_processed.drop(columns=['label', 'attack_label'])
y = df_processed['attack_label']

# Store feature column names for later use
feature_columns = X.columns.tolist()

# Scale the features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Shape of features after preprocessing: {X_scaled.shape}")
print(f"Number of features: {len(feature_columns)}")
print("First 5 rows of scaled features (sample):")
print(X_scaled[:5])

Shape of features after preprocessing: (80000, 1503)
Number of features: 1503
First 5 rows of scaled features (sample):
[[ 1.07399036  1.75656384 -0.7302174  ... -0.473423   -0.52148541
  -0.0314382 ]
 [-0.89842643 -1.01992409 -0.70043163 ... -0.473423   -0.52032358
  -0.51430923]
 [-1.21346523 -1.01992409  2.88875402 ... -1.23796633 -0.50638168
  -0.0314382 ]
 [ 1.60818658  1.96710321 -0.7302174  ... -0.473423   -0.53194183
  -0.0314382 ]
 [ 1.67667327  2.24343613  1.1462863  ... -1.3932642   1.47336838
  -0.51430923]]


In [19]:
# Re-executing Isolation Forest training with updated hyperparameters on the full dataset
# (Cell 214d30db)
# Train Isolation Forest (Anomaly Detection)
print("\nTraining Isolation Forest...")
# Update contamination based on the balanced dataset (40k normal, 40k attack => 0.5 anomaly ratio)
isolation_forest = IsolationForest(contamination=0.5, random_state=42, n_jobs=-1) # n_jobs=-1 for parallel processing
isolation_forest.fit(X_scaled)
print("Isolation Forest training complete.")

# Isolation Forest predictions (returns -1 for anomalies, 1 for inliers)
# Convert to 1 for anomaly, 0 for normal for consistency with attack_label
isolation_forest_predictions = np.array([1 if x == -1 else 0 for x in isolation_forest.predict(X_scaled)])

print("\nIsolation Forest Anomaly Detection Report:")
print(classification_report(y, isolation_forest_predictions))
print(f"Accuracy: {accuracy_score(y, isolation_forest_predictions):.4f}")


Training Isolation Forest...
Isolation Forest training complete.

Isolation Forest Anomaly Detection Report:
              precision    recall  f1-score   support

           0       0.44      0.44      0.44     40000
           1       0.44      0.44      0.44     40000

    accuracy                           0.44     80000
   macro avg       0.44      0.44      0.44     80000
weighted avg       0.44      0.44      0.44     80000

Accuracy: 0.4440


In [20]:
# Re-executing Random Forest training with updated hyperparameters on the full dataset
# (Cell 2c7de6fd)
# Train Random Forest Classifier (Attack Classifier)
print("\nTraining Random Forest Classifier (initial for feature importance)...")
# Update hyperparameters for better accuracy and for feature importance calculation
random_forest = RandomForestClassifier(n_estimators=150, max_depth=15, min_samples_split=2, random_state=42, n_jobs=-1) # n_jobs=-1 for parallel processing
random_forest.fit(X_scaled, y)
print("Random Forest Classifier training complete.")

# Random Forest predictions
random_forest_predictions = random_forest.predict(X_scaled)

print("\nRandom Forest Classifier Report (initial):")
print(classification_report(y, random_forest_predictions))
print(f"Accuracy: {accuracy_score(y, random_forest_predictions):.4f}")


Training Random Forest Classifier (initial for feature importance)...
Random Forest Classifier training complete.

Random Forest Classifier Report (initial):
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     40000
           1       0.99      1.00      0.99     40000

    accuracy                           0.99     80000
   macro avg       0.99      0.99      0.99     80000
weighted avg       0.99      0.99      0.99     80000

Accuracy: 0.9936


In [21]:
# Re-executing feature importance and selection on the full dataset
# (Cell ef1d733f)
# Get feature importances from the trained Random Forest model
importances = random_forest.feature_importances_
feature_names = X.columns

# Create a Series for easier sorting and filtering
feature_importances = pd.Series(importances, index=feature_names)

# Sort features by importance in descending order
sorted_features = feature_importances.sort_values(ascending=False)

print("Top 20 Most Important Features:")
print(sorted_features.head(20))

# Filter out features with 0 importance
active_features = sorted_features[sorted_features > 0].index.tolist()

print(f"\nNumber of active features (importance > 0): {len(active_features)}")
print("Active features will be used for retraining models.")

# Filter X to include only active features
X_selected = X[active_features]

# Re-scale features using a new scaler for only the active features
scaler_selected = StandardScaler()
X_selected_scaled = scaler_selected.fit_transform(X_selected)

print(f"Shape of selected features after scaling: {X_selected_scaled.shape}")

# Store selected feature column names for later use
selected_feature_columns = active_features

# Clear original large X_scaled to free memory
del X_scaled
gc.collect()

Top 20 Most Important Features:
ttl                0.107685
payload_byte_21    0.032193
payload_byte_28    0.028757
payload_byte_25    0.023394
payload_byte_17    0.023046
payload_byte_6     0.021516
payload_byte_20    0.019236
payload_byte_26    0.019014
total_len          0.018089
payload_byte_22    0.017570
payload_byte_3     0.017085
payload_byte_23    0.016694
payload_byte_32    0.016449
payload_byte_29    0.016389
payload_byte_30    0.016271
payload_byte_33    0.015966
payload_byte_7     0.015578
payload_byte_9     0.013083
payload_byte_14    0.013020
payload_byte_27    0.012731
dtype: float64

Number of active features (importance > 0): 1503
Active features will be used for retraining models.
Shape of selected features after scaling: (80000, 1503)


120

In [22]:
# Re-executing refined Isolation Forest training with selected features on the full dataset
# (Cell f8338129)
# Retrain Isolation Forest with selected features
print("\nRetraining Isolation Forest with selected features...")
isolation_forest_refined = IsolationForest(contamination=0.5, random_state=42, n_jobs=-1)
isolation_forest_refined.fit(X_selected_scaled)
print("Refined Isolation Forest training complete.")

isolation_forest_predictions_refined = np.array([1 if x == -1 else 0 for x in isolation_forest_refined.predict(X_selected_scaled)])

print("\nRefined Isolation Forest Anomaly Detection Report:")
print(classification_report(y, isolation_forest_predictions_refined))
print(f"Accuracy: {accuracy_score(y, isolation_forest_predictions_refined):.4f}")


Retraining Isolation Forest with selected features...
Refined Isolation Forest training complete.

Refined Isolation Forest Anomaly Detection Report:
              precision    recall  f1-score   support

           0       0.47      0.47      0.47     40000
           1       0.47      0.47      0.47     40000

    accuracy                           0.47     80000
   macro avg       0.47      0.47      0.47     80000
weighted avg       0.47      0.47      0.47     80000

Accuracy: 0.4706


In [23]:
# Re-executing refined Random Forest training with selected features on the full dataset
# (Cell 0fdaa707)
# Retrain Random Forest Classifier with selected features
print("\nRetraining Random Forest Classifier with selected features...")
random_forest_refined = RandomForestClassifier(n_estimators=150, max_depth=15, min_samples_split=2, random_state=42, n_jobs=-1)
random_forest_refined.fit(X_selected_scaled, y)
print("Refined Random Forest Classifier training complete.")

random_forest_predictions_refined = random_forest_refined.predict(X_selected_scaled)

print("\nRefined Random Forest Classifier Report:")
print(classification_report(y, random_forest_predictions_refined))
print(f"Accuracy: {accuracy_score(y, random_forest_predictions_refined):.4f}")


Retraining Random Forest Classifier with selected features...
Refined Random Forest Classifier training complete.

Refined Random Forest Classifier Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     40000
           1       0.99      1.00      0.99     40000

    accuracy                           0.99     80000
   macro avg       0.99      0.99      0.99     80000
weighted avg       0.99      0.99      0.99     80000

Accuracy: 0.9931


In [24]:
# Execute cell 9aee8e9b to re-process and scale features based on the 80,000-record df_combined
# This cell was modified in the previous step and needs to be re-run.
# The actual code content for this cell is the same as the modified_cells in the previous turn
# and is not explicitly regenerated here, as its execution is the key.
# For the purpose of this output, I am indicating the need to execute it.

In [25]:
# Re-execute Isolation Forest training with updated hyperparameters on the full dataset
# (Cell 214d30db)
# Train Isolation Forest (Anomaly Detection)
print("\nTraining Isolation Forest...")
# Update contamination based on the balanced dataset (40k normal, 40k attack => 0.5 anomaly ratio)
isolation_forest = IsolationForest(contamination=0.5, random_state=42, n_jobs=-1) # n_jobs=-1 for parallel processing
isolation_forest.fit(X_selected_scaled)
print("Isolation Forest training complete.")

# Isolation Forest predictions (returns -1 for anomalies, 1 for inliers)
# Convert to 1 for anomaly, 0 for normal for consistency with attack_label
isolation_forest_predictions = np.array([1 if x == -1 else 0 for x in isolation_forest.predict(X_selected_scaled)])

print("\nIsolation Forest Anomaly Detection Report:")
print(classification_report(y, isolation_forest_predictions))
print(f"Accuracy: {accuracy_score(y, isolation_forest_predictions):.4f}")


Training Isolation Forest...
Isolation Forest training complete.

Isolation Forest Anomaly Detection Report:
              precision    recall  f1-score   support

           0       0.47      0.47      0.47     40000
           1       0.47      0.47      0.47     40000

    accuracy                           0.47     80000
   macro avg       0.47      0.47      0.47     80000
weighted avg       0.47      0.47      0.47     80000

Accuracy: 0.4706


In [26]:
# Re-execute Random Forest training with updated hyperparameters on the full dataset
# (Cell 2c7de6fd)
# Train Random Forest Classifier (Attack Classifier)
print("\nTraining Random Forest Classifier (initial for feature importance)...")
# Update hyperparameters for better accuracy and for feature importance calculation
random_forest = RandomForestClassifier(n_estimators=150, max_depth=15, min_samples_split=2, random_state=42, n_jobs=-1) # n_jobs=-1 for parallel processing
random_forest.fit(X_selected_scaled, y)
print("Random Forest Classifier training complete.")

# Random Forest predictions
random_forest_predictions = random_forest.predict(X_selected_scaled)

print("\nRandom Forest Classifier Report (initial):")
print(classification_report(y, random_forest_predictions))
print(f"Accuracy: {accuracy_score(y, random_forest_predictions):.4f}")


Training Random Forest Classifier (initial for feature importance)...
Random Forest Classifier training complete.

Random Forest Classifier Report (initial):
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     40000
           1       0.99      1.00      0.99     40000

    accuracy                           0.99     80000
   macro avg       0.99      0.99      0.99     80000
weighted avg       0.99      0.99      0.99     80000

Accuracy: 0.9931


In [27]:
# Re-execute feature importance and selection on the full dataset
# (Cell ef1d733f)
# Get feature importances from the trained Random Forest model
importances = random_forest.feature_importances_
feature_names = X.columns

# Create a Series for easier sorting and filtering
feature_importances = pd.Series(importances, index=feature_names)

# Sort features by importance in descending order
sorted_features = feature_importances.sort_values(ascending=False)

print("Top 20 Most Important Features:")
print(sorted_features.head(20))

# Filter out features with 0 importance
active_features = sorted_features[sorted_features > 0].index.tolist()

print(f"\nNumber of active features (importance > 0): {len(active_features)}")
print("Active features will be used for retraining models.")

# Filter X to include only active features
X_selected = X[active_features]

# Re-scale features using a new scaler for only the active features
scaler_selected = StandardScaler()
X_selected_scaled = scaler_selected.fit_transform(X_selected)

print(f"Shape of selected features after scaling: {X_selected_scaled.shape}")

# Store selected feature column names for later use
selected_feature_columns = active_features

# Clear original large X_scaled to free memory, if it exists
if 'X_scaled' in locals():
    del X_scaled
gc.collect()

Top 20 Most Important Features:
payload_byte_1     0.098446
payload_byte_3     0.031678
payload_byte_14    0.031139
payload_byte_4     0.026712
payload_byte_2     0.023450
payload_byte_8     0.022654
payload_byte_5     0.021001
payload_byte_10    0.020704
payload_byte_11    0.018365
payload_byte_16    0.018111
payload_byte_12    0.018058
payload_byte_6     0.016379
payload_byte_9     0.016025
payload_byte_15    0.015491
payload_byte_7     0.015243
payload_byte_32    0.014531
payload_byte_20    0.013514
payload_byte_13    0.012862
payload_byte_27    0.011819
payload_byte_19    0.011601
dtype: float64

Number of active features (importance > 0): 1503
Active features will be used for retraining models.
Shape of selected features after scaling: (80000, 1503)


120

In [28]:
# Re-execute refined Isolation Forest training with selected features on the full dataset
# (Cell f8338129)
# Retrain Isolation Forest with selected features
print("\nRetraining Isolation Forest with selected features...")
isolation_forest_refined = IsolationForest(contamination=0.5, random_state=42, n_jobs=-1)
isolation_forest_refined.fit(X_selected_scaled)
print("Refined Isolation Forest training complete.")

isolation_forest_predictions_refined = np.array([1 if x == -1 else 0 for x in isolation_forest_refined.predict(X_selected_scaled)])

print("\nRefined Isolation Forest Anomaly Detection Report:")
print(classification_report(y, isolation_forest_predictions_refined))
print(f"Accuracy: {accuracy_score(y, isolation_forest_predictions_refined):.4f}")


Retraining Isolation Forest with selected features...
Refined Isolation Forest training complete.

Refined Isolation Forest Anomaly Detection Report:
              precision    recall  f1-score   support

           0       0.44      0.44      0.44     40000
           1       0.44      0.44      0.44     40000

    accuracy                           0.44     80000
   macro avg       0.44      0.44      0.44     80000
weighted avg       0.44      0.44      0.44     80000

Accuracy: 0.4413


In [29]:
# Re-execute refined Random Forest training with selected features on the full dataset
# (Cell 0fdaa707)
# Retrain Random Forest Classifier with selected features
print("\nRetraining Random Forest Classifier with selected features...")
random_forest_refined = RandomForestClassifier(n_estimators=150, max_depth=15, min_samples_split=2, random_state=42, n_jobs=-1)
random_forest_refined.fit(X_selected_scaled, y)
print("Refined Random Forest Classifier training complete.")

random_forest_predictions_refined = random_forest_refined.predict(X_selected_scaled)

print("\nRefined Random Forest Classifier Report:")
print(classification_report(y, random_forest_predictions_refined))
print(f"Accuracy: {accuracy_score(y, random_forest_predictions_refined):.4f}")


Retraining Random Forest Classifier with selected features...
Refined Random Forest Classifier training complete.

Refined Random Forest Classifier Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     40000
           1       0.99      1.00      0.99     40000

    accuracy                           0.99     80000
   macro avg       0.99      0.99      0.99     80000
weighted avg       0.99      0.99      0.99     80000

Accuracy: 0.9940


### How to access your locally saved model artifacts:

Now that the model artifacts are saved locally in your Colab environment, you have a couple of options to retrieve them:

1.  **Download to your local machine:**
    *   In the left sidebar of Colab, click on the **'Files'** icon (folder icon).
    *   You should see the `.pkl` files listed (e.g., `isolation_forest_refined.pkl`, `random_forest_refined.pkl`, `scaler_refined.pkl`, etc.).
    *   Right-click on each file and select **'Download'**.

2.  **Save to Google Drive:**
    *   You can also move these files to your Google Drive for persistent storage. You'll need to mount your Google Drive first.
    *   Run the following Python code in a new cell:

        ```python
        from google.colab import drive
        drive.mount('/content/drive')

        import os
        import shutil

        # Define the local directory where artifacts are saved
        local_artifact_dir = '/content/'
        # Define your desired Google Drive path
        drive_artifact_dir = '/content/drive/MyDrive/Colab_Model_Artifacts/' # You can change this path

        os.makedirs(drive_artifact_dir, exist_ok=True)

        artifacts = [
            "isolation_forest_refined.pkl",
            "random_forest_refined.pkl",
            "scaler_refined.pkl",
            "feature_columns_refined.pkl",
            "attack_classes.pkl",
            "label_encoder_refined.pkl"
        ]

        for artifact_name in artifacts:
            local_path = os.path.join(local_artifact_dir, artifact_name)
            drive_path = os.path.join(drive_artifact_dir, artifact_name)
            if os.path.exists(local_path):
                shutil.copy(local_path, drive_path)
                print(f"Copied {artifact_name} to Google Drive: {drive_path}")
            else:
                print(f"Artifact {artifact_name} not found locally.")
        ```

    *   After running this code, you'll find your model artifacts in the specified folder in your Google Drive.

### 3. Train two models (with updated hyperparameters)

We will now train an Isolation Forest model for anomaly detection and a Random Forest Classifier for attack classification with updated hyperparameters and on the larger dataset.

In [31]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# Fix: Ensure X_scaled is available. Re-create it if it was deleted.
# Assuming X and y are available from previous cells, and StandardScaler is imported.
if 'X_scaled' not in locals() and 'X_scaled' not in globals():
    print("X_scaled not found. Re-creating X_scaled from X and a new StandardScaler.")
    # Ensure X is a DataFrame and extract its columns for the scaler
    # (X is already confirmed as a DataFrame in kernel state)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    print(f"X_scaled re-created with shape: {X_scaled.shape}")

# Train Isolation Forest (Anomaly Detection)
print("\nTraining Isolation Forest...")
# Update contamination based on the balanced dataset (40k normal, 40k attack => 0.5 anomaly ratio)
isolation_forest = IsolationForest(contamination=0.5, random_state=42, n_jobs=-1) # n_jobs=-1 for parallel processing
isolation_forest.fit(X_scaled)
print("Isolation Forest training complete.")

# Isolation Forest predictions (returns -1 for anomalies, 1 for inliers)
# Convert to 1 for anomaly, 0 for normal for consistency with attack_label
isolation_forest_predictions = np.array([1 if x == -1 else 0 for x in isolation_forest.predict(X_scaled)])

print("\nIsolation Forest Anomaly Detection Report:")
print(classification_report(y, isolation_forest_predictions))
print(f"Accuracy: {accuracy_score(y, isolation_forest_predictions):.4f}")

X_scaled not found. Re-creating X_scaled from X and a new StandardScaler.
X_scaled re-created with shape: (80000, 1503)

Training Isolation Forest...
Isolation Forest training complete.

Isolation Forest Anomaly Detection Report:
              precision    recall  f1-score   support

           0       0.44      0.44      0.44     40000
           1       0.44      0.44      0.44     40000

    accuracy                           0.44     80000
   macro avg       0.44      0.44      0.44     80000
weighted avg       0.44      0.44      0.44     80000

Accuracy: 0.4440


In [ ]:
# Train Random Forest Classifier (Attack Classifier)
print("\nTraining Random Forest Classifier (initial for feature importance)...")
# Update hyperparameters for better accuracy and for feature importance calculation
random_forest = RandomForestClassifier(n_estimators=150, max_depth=15, min_samples_split=2, random_state=42, n_jobs=-1) # n_jobs=-1 for parallel processing
random_forest.fit(X_scaled, y)
print("Random Forest Classifier training complete.")

# Random Forest predictions
random_forest_predictions = random_forest.predict(X_scaled)

print("\nRandom Forest Classifier Report (initial):")
print(classification_report(y, random_forest_predictions))
print(f"Accuracy: {accuracy_score(y, random_forest_predictions):.4f}")

### 4. Feature Selection using Random Forest Importance

Now, we'll identify the top 20 most important features from the trained Random Forest model and filter out features with zero importance. We will then retrain both models using only these selected features.

In [32]:
# Get feature importances from the trained Random Forest model
importances = random_forest.feature_importances_
feature_names = X.columns

# Create a Series for easier sorting and filtering
feature_importances = pd.Series(importances, index=feature_names)

# Sort features by importance in descending order
sorted_features = feature_importances.sort_values(ascending=False)

print("Top 20 Most Important Features:")
print(sorted_features.head(20))

# Filter out features with 0 importance
active_features = sorted_features[sorted_features > 0].index.tolist()

print(f"\nNumber of active features (importance > 0): {len(active_features)}")
print("Active features will be used for retraining models.")

# Filter X_scaled to include only active features
X_selected = X[active_features]

# Re-scale features using a new scaler for only the active features
scaler_selected = StandardScaler()
X_selected_scaled = scaler_selected.fit_transform(X_selected)

print(f"Shape of selected features after scaling: {X_selected_scaled.shape}")

# Store selected feature column names for later use
selected_feature_columns = active_features

# Clear original large X_scaled to free memory
del X_scaled
gc.collect()

Top 20 Most Important Features:
payload_byte_1     0.098446
payload_byte_3     0.031678
payload_byte_14    0.031139
payload_byte_4     0.026712
payload_byte_2     0.023450
payload_byte_8     0.022654
payload_byte_5     0.021001
payload_byte_10    0.020704
payload_byte_11    0.018365
payload_byte_16    0.018111
payload_byte_12    0.018058
payload_byte_6     0.016379
payload_byte_9     0.016025
payload_byte_15    0.015491
payload_byte_7     0.015243
payload_byte_32    0.014531
payload_byte_20    0.013514
payload_byte_13    0.012862
payload_byte_27    0.011819
payload_byte_19    0.011601
dtype: float64

Number of active features (importance > 0): 1503
Active features will be used for retraining models.
Shape of selected features after scaling: (80000, 1503)


96

### 5. Retrain Models with Selected Features

We will now retrain both the Isolation Forest and Random Forest models using only the selected, more important features to improve performance and local execution speed.

In [33]:
# Retrain Isolation Forest with selected features
print("\nRetraining Isolation Forest with selected features...")
isolation_forest_refined = IsolationForest(contamination=0.5, random_state=42, n_jobs=-1)
isolation_forest_refined.fit(X_selected_scaled)
print("Refined Isolation Forest training complete.")

isolation_forest_predictions_refined = np.array([1 if x == -1 else 0 for x in isolation_forest_refined.predict(X_selected_scaled)])

print("\nRefined Isolation Forest Anomaly Detection Report:")
print(classification_report(y, isolation_forest_predictions_refined))
print(f"Accuracy: {accuracy_score(y, isolation_forest_predictions_refined):.4f}")


Retraining Isolation Forest with selected features...
Refined Isolation Forest training complete.

Refined Isolation Forest Anomaly Detection Report:
              precision    recall  f1-score   support

           0       0.44      0.44      0.44     40000
           1       0.44      0.44      0.44     40000

    accuracy                           0.44     80000
   macro avg       0.44      0.44      0.44     80000
weighted avg       0.44      0.44      0.44     80000

Accuracy: 0.4413


In [34]:
# Retrain Random Forest Classifier with selected features
print("\nRetraining Random Forest Classifier with selected features...")
random_forest_refined = RandomForestClassifier(n_estimators=150, max_depth=15, min_samples_split=2, random_state=42, n_jobs=-1)
random_forest_refined.fit(X_selected_scaled, y)
print("Refined Random Forest Classifier training complete.")

random_forest_predictions_refined = random_forest_refined.predict(X_selected_scaled)

print("\nRefined Random Forest Classifier Report:")
print(classification_report(y, random_forest_predictions_refined))
print(f"Accuracy: {accuracy_score(y, random_forest_predictions_refined):.4f}")


Retraining Random Forest Classifier with selected features...
Refined Random Forest Classifier training complete.

Refined Random Forest Classifier Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     40000
           1       0.99      1.00      0.99     40000

    accuracy                           0.99     80000
   macro avg       0.99      0.99      0.99     80000
weighted avg       0.99      0.99      0.99     80000

Accuracy: 0.9940


### 6. Export and download refined trained artifacts

Finally, we will save the refined models, selected scaler, and feature information as pickle files and trigger their download.

In [35]:
artifacts_to_save_refined = {
    "isolation_forest_refined.pkl": isolation_forest_refined,
    "random_forest_refined.pkl": random_forest_refined,
    "scaler_refined.pkl": scaler_selected,
    "feature_columns_refined.pkl": selected_feature_columns,
    "attack_classes.pkl": y.unique().tolist() # Still [0, 1]
}

# Create a LabelEncoder that represents the 0/1 mapping for consistency
le_refined = LabelEncoder()
le_refined.fit([0, 1])
artifacts_to_save_refined["label_encoder_refined.pkl"] = le_refined

for filename, artifact in artifacts_to_save_refined.items():
    with open(filename, 'wb') as f:
        pickle.dump(artifact, f)
    print(f"Artifact saved locally: {filename}")

Artifact saved locally: isolation_forest_refined.pkl
Artifact saved locally: random_forest_refined.pkl
Artifact saved locally: scaler_refined.pkl
Artifact saved locally: feature_columns_refined.pkl
Artifact saved locally: attack_classes.pkl
Artifact saved locally: label_encoder_refined.pkl


In [36]:
from google.colab import files

# Assuming artifacts_to_save_refined is still in scope from cell 5503ea2a's execution
# which defines the filenames that were saved.

print("Initiating download for refined model artifacts...")
for filename in artifacts_to_save_refined.keys():
    try:
        files.download(filename)
        print(f"Successfully initiated download for: {filename}")
    except Exception as e:
        print(f"Error downloading {filename}: {e}")
print("All available refined model artifacts have been offered for download.")

Initiating download for refined model artifacts...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully initiated download for: isolation_forest_refined.pkl


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Successfully initiated download for: attack_classes.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully initiated download for: label_encoder_refined.pkl
All available refined model artifacts have been offered for download.


### 4. Export and download all trained artifacts

Finally, we will save the trained models, scaler, and feature information as pickle files and trigger their download.

## Dataset Explanation

This dataset consists of two CSV files, `Payload_data_UNSW.csv` and `Payload_data_CICIDS2017.csv`, which appear to contain labeled network traffic data. These datasets are commonly used in network intrusion detection research.

Each row in these DataFrames likely represents a network packet or a flow of packets, characterized by:

*   **`payload_byte_1` to `payload_byte_1500`**: These columns represent the raw byte values of the packet payload. Each column corresponds to a specific byte position within the payload. The values are typically numerical representations of the bytes (0-255).
*   **`ttl`**: Time To Live, a field in the IP header that indicates the maximum number of hops a packet can traverse before being discarded.
*   **`total_len`**: The total length of the IP packet.
*   **`protocol`**: The network protocol used (e.g., TCP, UDP, ICMP, others like 'pipe', 'ipip' as seen in the sample).
*   **`t_delta`**: Time delta, likely representing the time difference between consecutive packets or events within a flow.
*   **`label`**: This is the classification label for the network traffic, indicating whether it's normal ('BENIGN') or a specific type of attack/activity ('analysis', 'exploits', etc.).

`df_unsw` (from `Payload_data_UNSW.csv`) and `df_cicids` (from `Payload_data_CICIDS2017.csv`) are two distinct datasets that contain similar types of network traffic data, often used for training and evaluating intrusion detection systems.